In [ ]:
%pip install -U bitsandbytes>=0.46.1

In [1]:
1+1

2

In [2]:
# Get loaddot_env and loging to huggingface
from dotenv import load_dotenv
import os
load_dotenv()
from huggingface_hub import login
# ( token: typing.Optional[str] = Noneadd_to_git_credential: bool = Falseskip_if_logged_in: bool = True )

token = os.getenv("HF_TOKEN")
login(token=token,skip_if_logged_in=True)


In [ ]:
model_id ="google/paligemma2-3b-pt-448"

from transformers import PaliGemmaProcessor, PaliGemmaForConditionalGeneration
from PIL import Image
import torch

# Load processor and model
processor = PaliGemmaProcessor.from_pretrained(model_id)
model = model = PaliGemmaForConditionalGeneration.from_pretrained(model_id, device_map="auto", torch_dtype=torch.bfloat16)



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# image_path = '/content/drive/MyDrive/testpaligemma-ios.jpeg'
image_path = 'testpaligemma-ios.jpeg'
image = Image.open(image_path).convert("RGB")
plt.imshow(image)
plt.axis('off')
plt.show()

In [ ]:
prompt = "<image>what is the date printed on the ticket?"

inputs = processor(text=prompt, images=image, return_tensors="pt").to(torch.bfloat16).to(model.device)
output = model.generate(**inputs, max_new_tokens=20)

print(processor.decode(output[0], skip_special_tokens=True))

In [ ]:
prompt = "<image>what is the total price on the ticket?"

inputs = processor(text=prompt, images=image, return_tensors="pt").to(torch.bfloat16).to(model.device)
output = model.generate(**inputs, max_new_tokens=20)

print(processor.decode(output[0], skip_special_tokens=True))

In [3]:
import os
import torch
from datasets import load_dataset
from transformers import (
    PaliGemmaForConditionalGeneration,
    PaliGemmaProcessor,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ds = load_dataset("merve/vqav2-small", split="validation")
split_ds = ds.train_test_split(test_size=0.95)
train_ds = split_ds["test"]

model_id = "google/paligemma2-3b-pt-448"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,  # safer on T4 than bf16 in many setups
)

model = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

# Freeze vision encoder + projector
for p in model.model.vision_tower.parameters():
    p.requires_grad = False

for p in model.model.multi_modal_projector.parameters():
    p.requires_grad = False

model = prepare_model_for_kbit_training(model)

model.gradient_checkpointing_enable()
model.config.use_cache = False

lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

processor = PaliGemmaProcessor.from_pretrained(model_id, use_fast=True)

def collate_fn(examples):
    texts = ["<image>answer en " + ex["question"] for ex in examples]
    labels = [ex["multiple_choice_answer"] for ex in examples]
    images = [ex["image"].convert("RGB") for ex in examples]

    batch = processor(
        text=texts,
        images=images,
        suffix=labels,
        return_tensors="pt",
        padding="longest",
    )

    return {k: v.to(model.device) for k, v in batch.items()}

args = TrainingArguments(
    output_dir="paligemma2_qlora_t4_minram",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    remove_unused_columns=False,
    learning_rate=1e-4,
    warmup_steps=5,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    dataloader_pin_memory=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
)

trainer = Trainer(
    model=model,
    train_dataset=train_ds,
    data_collator=collate_fn,
    args=args,
)

trainer.train()

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

/home/hero/ML_journey/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 1,296,384 || all params: 3,034,423,536 || trainable%: 0.0427


The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.
`SiglipImageProcessor` requires torchvision (not installed); falling back to `SiglipImageProcessorPil` for backward compatibility. Install torchvision to use the default backend, or import `SiglipImageProcessorPil` directly to silence this warning.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,1.201172
20,1.238909
30,1.356026
40,1.303889
50,0.736992
60,0.941983
70,1.050393
80,0.706719
90,0.870793
100,0.891704


KeyboardInterrupt: 

In [ ]:
# import torch
# from transformers import BitsAndBytesConfig, PaliGemmaForConditionalGeneration

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_compute_dtype=torch.float16,   # better choice on T4
# )

# model = PaliGemmaForConditionalGeneration.from_pretrained(
#     "google/paligemma2-3b-pt-448",
#     quantization_config=bnb_config,
#     device_map="auto",
# )